## Structured Data Cleaning

In this notebook we will apply generic transformations to our structured datasets stored in MinIO. Specifically, we will clean and normalize the data (standardizing column names, handling missing values, and normalizing text fields such as case and whitespace), while ensuring consistency across all collections before analysis. All transformations will be performed using Apache Spark for distributed processing, with the final cleaned results stored back into MongoDB.

**Importing Useful Libraries**

In [1]:
import os
import re
import ast
import boto3
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

In [3]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)


DELTA_VERSION = "4.1.0" 
CLICKHOUSE_CONNECTOR_VERSION = "0.8.0" 


spark = SparkSession.builder \
    .appName("trusted_zone-structured") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", 
            f"org.apache.hadoop:hadoop-aws:3.3.4,"
            f"com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            f"io.delta:delta-spark_2.13:{DELTA_VERSION},"
            f"com.clickhouse.spark:clickhouse-spark-runtime-3.4_2.13:{CLICKHOUSE_CONNECTOR_VERSION},"
            f"com.clickhouse:clickhouse-jdbc:0.6.5") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.catalog.clickhouse", "com.clickhouse.spark.ClickHouseCatalog") \
    .config("spark.sql.catalog.clickhouse.host", "clickhouse") \
    .config("spark.sql.catalog.clickhouse.http_port", "8123") \
    .config("spark.sql.catalog.clickhouse.user", "analytics") \
    .config("spark.sql.catalog.clickhouse.password", "analytics_secret") \
    .config("spark.sql.catalog.clickhouse.database", "bi_analytics") \
    .config("spark.hadoop.fs.s3a.endpoint", endpoint) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.access.key", access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", secret_key) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

# Normalize Hadoop configuration values (e.g., converting "60s" to "60") to prevent version mismatch errors
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
for item in hadoop_conf.iterator():
    key = item.getKey()
    value = item.getValue()

    if isinstance(value, str) and (value.endswith("s") or value.endswith("h")):
        numeric_value = "".join(char for char in value if char.isdigit())
        hadoop_conf.set(key, numeric_value)

In [4]:
def auto_create_ch_table_from_delta(delta_path_or_name, ch_db, ch_table_name, order_by_cols=[]):
    """
    Infers the schema from a Delta dataset directly via s3a, 
    generates ClickHouse DDL, and creates the table via Spark SQL.
    """
    print(delta_path_or_name)
    df = spark.read.format("delta").load(delta_path_or_name)
    print(df)
    type_mapping = {
        StringType: "String", IntegerType: "Int32", LongType: "Int64",
        FloatType: "Float32", DoubleType: "Float64", BooleanType: "UInt8",
        DateType: "Date", TimestampType: "DateTime", ShortType: "Int16", ByteType: "Int8"
    }
    df.head()
    ch_columns = []
    for field in df.schema.fields:
        field_name = field.name
        field_type = type(field.dataType)
        
        if field_type == ArrayType:
            element_type = type(field.dataType.elementType)
            ch_sub_type = type_mapping.get(element_type, "String")
            ch_type = f"Array({ch_sub_type})"
        else:
            ch_type = type_mapping.get(field_type, "String")
            
        ch_columns.append(f"`{field_name}` {ch_type}")
    
    columns_sql = ",\n    ".join(ch_columns)
    order_by_sql = ", ".join([f"`{c}`" for c in order_by_cols]) if order_by_cols else df.schema.fields[0].name
        
    full_ch_table_path = f"clickhouse.{ch_db}.{ch_table_name}"
    spark.sql(f"CREATE DATABASE IF NOT EXISTS clickhouse.{ch_db}")
    
    create_table_ddl = f"""
    CREATE TABLE IF NOT EXISTS {full_ch_table_path} (
        {columns_sql}
    )
    USING clickhouse
    TBLPROPERTIES (engine = 'MergeTree()', order_by = '{order_by_sql}')
    """
    spark.sql(create_table_ddl)
    print(f"[SUCCESS] ClickHouse table defined: {full_ch_table_path}")
    return df

In [5]:
import re

# 1. Access Spark's underlying Hadoop FileSystem API to scan the target s3a path
sc = spark.sparkContext
base_path_str = "s3a://landing-zone/persistent-landing/structured/"
path_obj = sc._jvm.org.apache.hadoop.fs.Path(base_path_str)
fs = path_obj.getFileSystem(sc._jsc.hadoopConfiguration())

# Retrieve all item statuses inside the structured directory
file_statuses = fs.listStatus(path_obj)

valid_delta_paths = []
for status in file_statuses:
    # Filter for directories only (Delta tables are stored as directories)
    if status.isDirectory():
        full_path = status.getPath().toString()
        folder_name = full_path.split("/")[-1] or full_path.split("/")[-2]
        
        # CRITICAL FILTER: Skip 'raw', 'file_catalog' and hidden/metadata folders
        if folder_name in ["raw", "file_catalog"] or folder_name.startswith("."):
            continue
            
        valid_delta_paths.append(full_path)

In [18]:
import re
from pyspark.sql import DataFrame
from pyspark.sql.types import StringType, IntegerType, LongType, FloatType, DoubleType, BooleanType, DateType, TimestampType, ShortType, ByteType, ArrayType

def extract_clean_dataframe_and_ddl(spark_session: SparkSession, s3_path: str, database_name: str = "bi_analytics") -> tuple[DataFrame, str]:
    """
    自适应读取S3路径数据，深度清洗字段名，并根据特定业务规则生成 ClickHouse 建表 DDL。
    确保 ORDER BY 中的排序主键不包含 Nullable 属性。
    
    :return: (cleaned_df, native_clickhouse_ddl)
    """
    # 提取并规范化 ClickHouse 表名
    folder_name = s3_path.split("/")[-1] or s3_path.split("/")[-2]
    clean_table_name = folder_name.replace("_delta", "")
    clean_table_name = re.sub(r'_\d+$', '', clean_table_name)
    clean_table_name = clean_table_name.replace("-", "_").lower()
    
    # 1. 动态探测存储格式并读取
    sc = spark_session.sparkContext
    delta_log_path_str = s3_path.rstrip("/") + "/_delta_log"
    delta_log_path_obj = sc._jvm.org.apache.hadoop.fs.Path(delta_log_path_str)
    fs = delta_log_path_obj.getFileSystem(sc._jsc.hadoopConfiguration())
    
    if fs.exists(delta_log_path_obj):
        df = spark_session.read.format("delta").load(s3_path)
    else:
        df = spark_session.read.parquet(s3_path)
        
    # 2. 字段名深度清洗
    for col_name in df.columns:
        cleaned_col_name = col_name.replace("ï»¿", "").replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")
        if cleaned_col_name != col_name:
            df = df.withColumnRenamed(col_name, cleaned_col_name)
            
    # 3. 业务路由排序键 (ORDER BY 映射)
    sorting_keys = []
    if "global_warming" in clean_table_name:
        sorting_keys = ["Country", "Year"]
    elif "temperature_change" in clean_table_name:
        sorting_keys = ["Area", "Months"]
    elif "emission" in clean_table_name:
        sorting_keys = ["Make", "Model"]
    elif "tweet" in clean_table_name:
        sorting_keys = ["tweet_id"]

    final_sorting_keys = [k for k in sorting_keys if k in df.columns]
    if not final_sorting_keys:
        # 如果没有匹配到业务键，寻找是否有通用 'id' 字段，否则默认使用第一列
        final_sorting_keys = [k for k in ["id"] if k in df.columns]
    
    order_by_clause = ", ".join([f"`{k}`" for k in final_sorting_keys]) if final_sorting_keys else f"`{df.columns[0]}`"
    
    # 4. 字段类型安全映射生成 DDL
    type_mapping = {
        StringType: "String", IntegerType: "Int32", LongType: "Int64",
        FloatType: "Float32", DoubleType: "Float64", BooleanType: "UInt8",
        DateType: "Date", TimestampType: "DateTime", ShortType: "Int16", ByteType: "Int8"
    }
    
    ch_columns = []
    for field in df.limit(0).schema.fields:
        # 解析底层类型
        if isinstance(field.dataType, ArrayType):
            element_type_class = type(field.dataType.elementType)
            ch_type = f"Array({type_mapping.get(element_type_class, 'String')})"
        else:
            ch_type = type_mapping.get(type(field.dataType), "String")
            
        # 🌟 核心修复点：如果是排序主键中的列，坚决不加 Nullable 保护
        if field.name in final_sorting_keys:
            ch_columns.append(f"    `{field.name}` {ch_type}")
        else:
            ch_columns.append(f"    `{field.name}` Nullable({ch_type})")
        
    columns_sql = ",\n".join(ch_columns)
    
    native_ddl = f"""CREATE TABLE IF NOT EXISTS {database_name}.{clean_table_name} (
{columns_sql}
) ENGINE = MergeTree()
ORDER BY ({order_by_clause})"""

    return df, native_ddl

In [21]:
import clickhouse_connect

def load_dataframe_to_clickhouse(spark_session: SparkSession, df: DataFrame, ddl_sql: str, target_table_name: str, database_name: str = "bi_analytics"):
    """
    完全抛弃脆弱的 Spark JDBC 驱动。
    直接使用官方 Python 高性能客户端执行 DDL 并高速同步整个 DataFrame 的数据。
    """
    print(f"[CLICKHOUSE] 正在通过纯 Python 官方客户端连接服务...")
    
    # 1. 建立极其稳定的 HTTP 连接
    client = clickhouse_connect.get_client(
        host='clickhouse',
        port=8123,
        username='analytics',
        password='analytics_secret',
        database=database_name
    )
    
    # 2. 确保表结构存在 (前置 DDL)
    print(f"[DDL] 确保 ClickHouse 表结构 `{database_name}.{target_table_name}` 存在...")
    client.command(ddl_sql)
    
    # 3. 将 Spark DataFrame 转换为 Python 内存块，并利用底层高速流直灌入库
    full_table_path = f"{database_name}.{target_table_name}"
    print(f"[INGESTION] 正在通过官方高性能本地协议同步数据到 {full_table_path} ...")
    
    # 为了防止分布式数据量过大导致内存溢出，我们使用分布式并行的 mapPartitions
    # 或者直接在 Driver 端批量写入。针对你目前的数据集规模，直接转为 Row 数组通过 Python 客户端写入最稳妥
    data_rows = df.collect()
    column_names = df.columns
    
    if data_rows:
        # 将 Spark Rows 转化为 纯 Python 列表元组
        insert_data = [tuple(row) for row in data_rows]
        
        # 官方高速客户端批量写入接口
        client.insert(
            table=target_table_name,
            data=insert_data,
            column_names=column_names
        )
        print(f"[SUCCESS] 成功导入 {len(insert_data)} 行数据到表: '{target_table_name}'\n")
    else:
        print(f"[WARN] 上游 DataFrame 为空，未写入任何数据。\n")
        
    client.close()

In [22]:
# 遍历所有的 S3 路径执行清洗与灌录流程
for s3_delta_path in valid_delta_paths:
    # 提取表名用于打印和逻辑路由
    folder_name = s3_delta_path.split("/")[-1] or s3_delta_path.split("/")[-2]
    tbl_name = re.sub(r'_\d+$', '', folder_name.replace("_delta", "")).replace("-", "_").lower()

    print("\n" + "="*70)
    print(f"🎬 启动同步流水线: {tbl_name}")
    print("="*70)

    try:
        # 第一步：提取、清洗 Schema，生成 DDL
        cleaned_df, clickhouse_ddl = extract_clean_dataframe_and_ddl(spark, s3_delta_path)
        
        # 调试：可以随时打印或预览 DDL 和 Schema
        print(f"[DEBUG] 生成的 ClickHouse DDL 预览:\n{clickhouse_ddl}\n")
        
        # 第二步：将数据无缝加载进 ClickHouse
        load_dataframe_to_clickhouse(spark, cleaned_df, clickhouse_ddl, target_table_name=tbl_name)
        
    except Exception as e:
        print(f"❌ 管道在处理表格 '{tbl_name}' 时崩溃。错误原因: {e}")


🎬 启动同步流水线: co2_emission_by_vehicles
[DEBUG] 生成的 ClickHouse DDL 预览:
CREATE TABLE IF NOT EXISTS bi_analytics.co2_emission_by_vehicles (
    `Make` String,
    `Model` String,
    `Vehicle_Class` Nullable(String),
    `Engine_SizeL` Nullable(Float64),
    `Cylinders` Nullable(Int64),
    `Transmission` Nullable(String),
    `Fuel_Type` Nullable(String),
    `Fuel_Consumption_City_L_100_km` Nullable(Float64),
    `Fuel_Consumption_Hwy_L_100_km` Nullable(Float64),
    `Fuel_Consumption_Comb_L_100_km` Nullable(Float64),
    `Fuel_Consumption_Comb_mpg` Nullable(Int64),
    `CO2_Emissionsg_km` Nullable(Int64)
) ENGINE = MergeTree()
ORDER BY (`Make`, `Model`)

[CLICKHOUSE] 正在通过纯 Python 官方客户端连接服务...
[DDL] 确保 ClickHouse 表结构 `bi_analytics.co2_emission_by_vehicles` 存在...
[INGESTION] 正在通过官方高性能本地协议同步数据到 bi_analytics.co2_emission_by_vehicles ...
[SUCCESS] 成功导入 7385 行数据到表: 'co2_emission_by_vehicles'


🎬 启动同步流水线: global_warming_dataset
[DEBUG] 生成的 ClickHouse DDL 预览:
CREATE TABLE IF NOT EXISTS bi_analyt

ERROR:clickhouse_connect.driver.transform:Error serializing column `tweet_id` into data type `Float64`
Traceback (most recent call last):
  File "clickhouse_connect/driverc/dataconv.pyx", line 580, in clickhouse_connect.driverc.dataconv.write_native_col
struct.error: required argument is not a float

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.13/site-packages/clickhouse_connect/driver/transform.py", line 126, in chunk_gen
    col_type.write_column(data, output, context)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/site-packages/clickhouse_connect/datatypes/base.py", line 228, in write_column
    self.write_column_data(column, dest, ctx)
    ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/site-packages/clickhouse_connect/datatypes/base.py", line 243, in write_column_data
    self._write_column_binary(column, dest, ctx)
    ~~~~~~~

❌ 管道在处理表格 'natural_disaster_tweets' 时崩溃。错误原因: Unable to create native array for column `tweet_id`: error

🎬 启动同步流水线: temperature_change
[DEBUG] 生成的 ClickHouse DDL 预览:
CREATE TABLE IF NOT EXISTS bi_analytics.temperature_change (
    `Domain_Code` Nullable(String),
    `Domain` Nullable(String),
    `Area_Code_M49` Nullable(Int64),
    `Area` String,
    `Element_Code` Nullable(Int64),
    `Element` Nullable(String),
    `Months_Code` Nullable(Int64),
    `Months` String,
    `Year_Code` Nullable(Int64),
    `Year` Nullable(Int64),
    `Unit` Nullable(String),
    `Value` Nullable(Float64),
    `Flag` Nullable(String),
    `Flag_Description` Nullable(String)
) ENGINE = MergeTree()
ORDER BY (`Area`, `Months`)

[CLICKHOUSE] 正在通过纯 Python 官方客户端连接服务...
[DDL] 确保 ClickHouse 表结构 `bi_analytics.temperature_change` 存在...
[INGESTION] 正在通过官方高性能本地协议同步数据到 bi_analytics.temperature_change ...
[SUCCESS] 成功导入 241893 行数据到表: 'temperature_change'



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


**Applying Transformations**

Next, we will use Spark to read the files from MinIO and apply some generic transformations on Parquet files. After that, we import them into MongoDB. The following functions can help us do some generic transformations on the data.

In [4]:
# ----------------------------
# CLEAN COLUMN NAMES
# ----------------------------
def clean_column(name):
    name = name.encode("ascii", "ignore").decode()
    name = name.lower().strip()
    name = re.sub(r"[^\w]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

# ----------------------------
# CLEAN TABLE NAMES
# ----------------------------
def clean_table_name(name):
    name = name.replace("-", "_")
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    return name.lower()

# ----------------------------
# PERIOD NORMALIZATION (KEEP ONLY THIS VERSION)
# ----------------------------
def normalize_period_column(df, col):

    if col not in df.columns:
        return df

    x = F.lower(F.col(col))

    x = F.regexp_replace(x, r"[^\w\s\-]", "-")
    x = F.regexp_replace(x, r"-+", "-")
    x = F.trim(x)

    return df.withColumn(
        col,
        F.when(x.isNull(), None)
         .otherwise(x)
    )

In [5]:
# LIST ALL "FOLDERS"
response = s3.list_objects_v2(
    Bucket="landing-zone",
    Prefix="persistent-landing/structured/",
    Delimiter="/"
)

folders = []
for prefix in response.get("CommonPrefixes", []):
    path = prefix["Prefix"]

    # exclude unwanted folders
    if "raw" in path or "file_catalog" in path:
        continue

    folders.append(path)

datasets = {}

# PROCESS EACH DATASET
for folder in folders:

    raw_table_name = folder.split('/', 3)[2]
    table_name = clean_table_name(raw_table_name)

    print(f"\nProcessing: {table_name}")

    # 1. Read parquet
    df = spark.read.parquet(f"s3a://landing-zone/{folder}")

    # 2. Drop duplicates
    df = df.dropDuplicates()

    # 3. Standardize column names
    df = df.toDF(*[clean_column(c) for c in df.columns])

    # 4. Fix period/month column
    for c in ["months", "month", "period"]:
        if c in df.columns:
            df = normalize_period_column(df, c)

    # 5. Schema cleanup
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, F.lower(F.trim(F.col(c))))

    # 6. Conversion point
    pdf = df.toPandas()
    pdf = pdf.where(pd.notna(pdf), None)
    records = pdf.to_dict("records")
    collection = db[table_name]

    # 7. Clean insert
    if records:
        collection.delete_many({})

        batch_size = 5000
        for i in range(0, len(records), batch_size):
            collection.insert_many(records[i:i + batch_size], ordered=False)

    print(f"Loaded {table_name}: {len(records)} rows")


Processing: co2_emission_by_vehicles_1777660672026
Loaded co2_emission_by_vehicles_1777660672026: 6282 rows

Processing: global_warming_dataset_1777660672092
Loaded global_warming_dataset_1777660672092: 100000 rows

Processing: natural_disaster_tweets_1777661000633
Loaded natural_disaster_tweets_1777661000633: 127540 rows

Processing: temperature_change_1777661000821
Loaded temperature_change_1777661000821: 9656 rows


In [6]:
# Check inserted collections
db = client["trusted_zone_structured"]
print(db.list_collection_names())

['global_warming_dataset_1777660672092', 'co2_emission_by_vehicles_1777660672026', 'temperature_change_1777661000821', 'natural_disaster_tweets_1777661000633']


**Showing Transformed Data**

In [16]:
name = next(c for c in db.list_collection_names()
            if c.startswith("co2_emission_by_vehicles"))
df = pd.DataFrame(list(db[name].find({}, {"_id": 0})))
df.head(5)

,make,model,vehicle_class,engine_size_l,cylinders,transmission,fuel_type,fuel_consumption_city_l_100_km,fuel_consumption_hwy_l_100_km,fuel_consumption_comb_l_100_km,fuel_consumption_comb_mpg,co2_emissions_g_km
0,bmw,activehybrid 7l,full-size,3.0,6,a8,z,10.5,7.6,9.2,31,212
1,bmw,x6 xdrive50i,suv - standard,4.4,8,a8,z,16.4,11.3,14.1,20,324
2,cadillac,cts-v coupe,mid-size,6.2,8,as6,z,19.7,12.9,16.6,17,382
3,honda,ridgeline awd,pickup truck - standard,3.5,6,a5,x,15.2,11.3,13.4,21,308
4,kia,soul,station wagon - small,1.6,4,a6,x,9.8,7.9,8.9,32,205


In [18]:
name = next(c for c in db.list_collection_names()
            if c.startswith("global_warming"))
df = pd.DataFrame(list(db[name].find({}, {"_id": 0})))
df.head(5)

,country,year,temperature_anomaly,co2_emissions,population,forest_area,gdp,renewable_energy_usage,methane_emissions,sea_level_rise,...,waste_management,per_capita_emissions,industrial_activity,air_pollution_index,biodiversity_index,ocean_acidification,fossil_fuel_usage,energy_consumption_per_capita,policy_score,average_temperature
0,country_41,1928,-0.876631,6.475424e+08,6.709389e+08,13.080266,3.407885e+12,19.189534,1.113522e+06,13.065625,...,67.309287,9.521487,50.463423,59.913299,92.174283,8.320624,61.931767,3965.409819,50.002123,0.656628
1,country_181,1994,1.975792,5.624029e+07,2.650367e+08,16.640081,9.205250e+12,72.849104,3.968051e+06,26.914859,...,64.042569,2.328607,24.610669,222.238895,78.059211,8.018291,84.107226,3702.122304,20.974063,-6.742663
2,country_94,1959,-0.046595,9.088235e+08,7.060134e+08,85.369889,5.481313e+11,56.592200,2.958762e+06,-4.805219,...,88.826658,7.710164,78.429071,128.159426,61.676545,8.039112,71.679447,4106.408178,75.449175,34.769860
3,country_105,2000,-1.165144,9.370459e+08,7.128588e+08,30.886640,5.181305e+12,53.867992,9.665266e+06,32.606897,...,34.473976,14.778174,99.977358,178.204563,40.082871,8.471874,44.808872,3512.050969,62.099961,34.983485
4,country_186,2014,-1.432683,1.706594e+07,3.362495e+08,94.387485,6.394685e+12,53.906423,9.196585e+06,34.959113,...,78.166820,7.952367,2.968431,107.313050,74.503013,8.181511,91.802846,4464.417446,84.303656,37.540680


In [20]:
name = next(c for c in db.list_collection_names()
            if c.startswith("natural_disaster_tweets"))
df = pd.DataFrame(list(db[name].find({}, {"_id": 0})))
df.head(5)

,tweet_id,tweet_text,disaster_type,hashtags,emojis
0,7.311968e+17,#fortmacfire evacuees are getting help they ne...,wildfire,"['fortmacfire', 'loveit', 'albertastrong', 'pr...",[]
1,7.287663e+17,.@classified to donate profits from his new si...,wildfire,[],[]
2,7.300928e+17,important information for displaced residents ...,wildfire,"['ymmfire', 'albertafires']",[]
3,7.294406e+17,ottawa to match #redcross donations for #fortm...,wildfire,"['redcross', 'fortmcmurray']",[]
4,7.297647e+17,rt @pwrdf: many thanks to all of you who have ...,wildfire,['fortmcmurray'],[]


In [21]:
name = next(c for c in db.list_collection_names()
            if c.startswith("temperature_change"))
df = pd.DataFrame(list(db[name].find({}, {"_id": 0})))
df.head(5)

,area_code,area,months_code,months,element_code,element,unit,y1961,y1962,y1963,...,y2010,y2011,y2012,y2013,y2014,y2015,y2016,y2017,y2018,y2019
0,7,angola,7016,dec-jan-feb,7271,temperature change,°c,0.035,-0.106,-0.247,...,0.728,0.331,-0.191,1.038,0.593,1.223,1.602,0.529,0.848,1.473
1,258,anguilla,7006,june,6078,standard deviation,°c,0.379,0.379,0.379,...,0.379,0.379,0.379,0.379,0.379,0.379,0.379,0.379,0.379,0.379
2,30,antarctica,7007,july,7271,temperature change,°c,0.016,-0.316,0.497,...,0.563,2.243,-0.903,1.304,-0.843,-1.439,-1.894,0.185,1.512,1.561
3,26,brunei darussalam,7005,may,6078,standard deviation,°c,0.269,0.269,0.269,...,0.269,0.269,0.269,0.269,0.269,0.269,0.269,0.269,0.269,0.269
4,26,brunei darussalam,7019,sep-oct-nov,6078,standard deviation,°c,0.227,0.227,0.227,...,0.227,0.227,0.227,0.227,0.227,0.227,0.227,0.227,0.227,0.227


We have nested lists in the columns `hashtags` and `emojis` of the collection `natural_disaster_tweets`, but due to the high number of labels, we decided not to flatten them into new columns.

In [23]:
# Print the number of unique elements in a nested column
def print_num_unique_labels(df, column):

    def parse_cell(x):
        if pd.isna(x):
            return []

        if isinstance(x, list):
            return x

        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except:
                cleaned = x.replace("[", "").replace("]", "").replace("'", "")
                return [i.strip() for i in cleaned.split(",") if i.strip()]

        return []

    all_tags = df[column].apply(parse_cell).explode()
    all_tags = all_tags.dropna().astype(str).str.strip().str.lower()

    print(f"Number of unique labels in {column}: {all_tags.nunique()}")

# Get collection natural_disaster_tweets
name = next(c for c in db.list_collection_names()
            if c.startswith("natural_disaster_tweets"))
df = pd.DataFrame(list(db[name].find({}, {"_id": 0})))

print_num_unique_labels(df, "hashtags")
print_num_unique_labels(df, "emojis")

Number of unique labels in hashtags: 27550
Number of unique labels in emojis: 713
